# Daily Dhan → BigQuery (thin notebook)

All logic lives in the `dhan-pipeline` package. This notebook just installs it, sets config, and runs.

In [ ]:
# 1. Install the shared package from GitHub (pin a tag/commit in production)
!pip install -q "git+https://github.com/<your-user>/dhan-pipeline.git"

In [ ]:
# 2. Mount Drive for the service-account JSON, then configure
import os
from google.colab import drive
drive.mount('/content/drive')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/content/drive/MyDrive/Colab Notebooks/rajat-trade-c411eaec7c51.json'

from google.colab import userdata
from dhan_pipeline import Config

cfg = Config(
    dhan_client_id=userdata.get('DHAN_CLIENT_ID'),
    dhan_access_token=userdata.get('DHAN_ACCESS_TOKEN'),
)

In [ ]:
# 3. Run: fetch last 2 days -> split-check 2nd-last day -> upsert last day
from dhan_pipeline.daily import run_daily

result = run_daily(cfg)
result['flags']  # scrips flagged for a suspected split/adjustment

In [ ]:
# 4. (Optional) subset example: only intraday=yes_1 scrips, custom window
from dhan_pipeline import load_scrip_mapping, gspread_client, subset
gc = gspread_client(cfg)
mapping = load_scrip_mapping(cfg, gc)
intraday = subset(mapping, 'intraday', ['yes_1'])
# run_daily(cfg, scrip_mapping=intraday)